# Q2 Data Cleaning

**Phase 3:** Data Cleaning & Preprocessing  
**Points: 9 points**

**Focus:** Handle missing data, outliers, validate data types, remove duplicates.

**Lecture Reference:** See **Lecture 11, Notebook 1** (`11/demo/01_setup_exploration_cleaning.ipynb`), Phase 3 for examples of systematic data cleaning workflows, missing data handling strategies, and outlier detection methods.

In [244]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Create output directory if it doesn't exist
os.makedirs('output', exist_ok=True)

# Load the original dataset
df = pd.read_csv('data/beach_sensors.csv')

In [245]:
# Store initial row count
rows_before = len(df)
print(f"\nRows before cleaning: {rows_before}")
print(f"Columns: {df.shape[1]}")


Rows before cleaning: 195892
Columns: 18


## Convert DateTime Variable

In [246]:
# Identify datetime column
datetime_col = 'Measurement Timestamp' 

# Convert datetime column
print(f"\nConverting '{datetime_col}' to datetime...")
df[datetime_col] = pd.to_datetime(df[datetime_col])
print(f"✓ {datetime_col} converted to datetime64[ns]")

# Verify numeric columns are numeric
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumeric columns identified: {len(numeric_cols)}")
for col in numeric_cols:
    print(f"  - {col}")

# Store data type conversions for report
data_type_conversions = [
    f"{datetime_col}: Converted to datetime64[ns]"
]


Converting 'Measurement Timestamp' to datetime...
✓ Measurement Timestamp converted to datetime64[ns]

Numeric columns identified: 14
  - Air Temperature
  - Wet Bulb Temperature
  - Humidity
  - Rain Intensity
  - Interval Rain
  - Total Rain
  - Precipitation Type
  - Wind Direction
  - Wind Speed
  - Maximum Wind Speed
  - Barometric Pressure
  - Solar Radiation
  - Heading
  - Battery Life


## Handling Missing Data

In [247]:
# Count missing values
missing_before = df.isnull().sum()
missing_pct = (missing_before / len(df)) * 100

print("\nMissing values by column:")
for col in df.columns:
    if missing_before[col] > 0:
        print(f"  - {col}: {missing_before[col]} ({missing_pct[col]:.2f}%)")

# Store missing data handling info for report
missing_data_report = []

# Handle missing data for each numeric column with missing values
for col in numeric_cols:
    if missing_before[col] > 0:
        print(f"\nHandling missing data in '{col}'...")
        
        # Strategy: Forward-fill for time series, then backward-fill, then median
        # This is appropriate for continuous sensor readings
        
        # Sort by datetime to ensure proper forward-fill
        df = df.sort_values(by=datetime_col)
        
        # Forward-fill (carry last observation forward)
        df[col] = df[col].ffill()
        
        # Backward-fill for any remaining NaN at the start
        df[col] = df[col].bfill()
        
        # If still any NaN, use median imputation
        remaining_na = df[col].isnull().sum()
        if remaining_na > 0:
            median_value = df[col].median()
            df[col] = df[col].fillna(median_value)
            method_used = f"Forward-fill, backward-fill, then median imputation ({median_value:.2f})"
        else:
            method_used = "Forward-fill and backward-fill"
        
        missing_data_report.append({
            'column': col,
            'count': int(missing_before[col]),
            'percentage': float(missing_pct[col]),
            'method': method_used
        })
        
        print(f"  ✓ {col}: {missing_before[col]} values handled using {method_used}")

# Verify no missing values remain in numeric columns
missing_after = df[numeric_cols].isnull().sum().sum()
print(f"\nTotal missing values in numeric columns after handling: {missing_after}")


Missing values by column:
  - Air Temperature: 75 (0.04%)
  - Wet Bulb Temperature: 75736 (38.66%)
  - Rain Intensity: 75736 (38.66%)
  - Total Rain: 75736 (38.66%)
  - Precipitation Type: 75736 (38.66%)
  - Barometric Pressure: 146 (0.07%)
  - Heading: 75736 (38.66%)

Handling missing data in 'Air Temperature'...
  ✓ Air Temperature: 75 values handled using Forward-fill and backward-fill

Handling missing data in 'Wet Bulb Temperature'...
  ✓ Wet Bulb Temperature: 75736 values handled using Forward-fill and backward-fill

Handling missing data in 'Rain Intensity'...
  ✓ Rain Intensity: 75736 values handled using Forward-fill and backward-fill

Handling missing data in 'Total Rain'...
  ✓ Total Rain: 75736 values handled using Forward-fill and backward-fill

Handling missing data in 'Precipitation Type'...
  ✓ Precipitation Type: 75736 values handled using Forward-fill and backward-fill

Handling missing data in 'Barometric Pressure'...
  ✓ Barometric Pressure: 146 values handled usin

## Handling Outliers

In [248]:
# Store outlier handling info for report
outlier_report = []

# Use IQR method for outlier detection (3×IQR is more conservative than 1.5×IQR)
for col in numeric_cols:
    print(f"\nAnalyzing outliers in '{col}'...")
    
    # Calculate IQR
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    # Define outlier bounds (using 3×IQR for conservative approach)
    lower_bound = Q1 - 3 * IQR
    upper_bound = Q3 + 3 * IQR
    
    # Count outliers
    outliers_mask = (df[col] < lower_bound) | (df[col] > upper_bound)
    outliers_count = outliers_mask.sum()
    
    print(f"  Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")
    print(f"  Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
    print(f"  Outliers detected: {outliers_count}")
    
    if outliers_count > 0:
        # Cap outliers at bounds (Winsorization)
        df.loc[df[col] < lower_bound, col] = lower_bound
        df.loc[df[col] > upper_bound, col] = upper_bound
        
        outlier_report.append({
            'column': col,
            'count': int(outliers_count),
            'method': 'Capped at IQR bounds (3×IQR)',
            'lower_bound': float(lower_bound),
            'upper_bound': float(upper_bound)
        })
        
        print(f"  ✓ {outliers_count} outliers capped to bounds")
    else:
        print(f"  ✓ No outliers detected")


Analyzing outliers in 'Air Temperature'...
  Q1: 4.39, Q3: 21.50, IQR: 17.11
  Bounds: [-46.94, 72.83]
  Outliers detected: 0
  ✓ No outliers detected

Analyzing outliers in 'Wet Bulb Temperature'...
  Q1: 2.80, Q3: 18.20, IQR: 15.40
  Bounds: [-43.40, 64.40]
  Outliers detected: 0
  ✓ No outliers detected

Analyzing outliers in 'Humidity'...
  Q1: 57.00, Q3: 80.00, IQR: 23.00
  Bounds: [-12.00, 149.00]
  Outliers detected: 0
  ✓ No outliers detected

Analyzing outliers in 'Rain Intensity'...
  Q1: 0.00, Q3: 0.00, IQR: 0.00
  Bounds: [0.00, 0.00]
  Outliers detected: 6851
  ✓ 6851 outliers capped to bounds

Analyzing outliers in 'Interval Rain'...
  Q1: 0.00, Q3: 0.00, IQR: 0.00
  Bounds: [0.00, 0.00]
  Outliers detected: 15798
  ✓ 15798 outliers capped to bounds

Analyzing outliers in 'Total Rain'...
  Q1: 15.70, Q3: 186.70, IQR: 171.00
  Bounds: [-497.30, 699.70]
  Outliers detected: 4569
  ✓ 4569 outliers capped to bounds

Analyzing outliers in 'Precipitation Type'...
  Q1: 0.00, Q

## Handling Duplicates

In [249]:
# Check for duplicate rows
duplicates_count = df.duplicated().sum()
print(f"\nDuplicate rows found: {duplicates_count}")

if duplicates_count > 0:
    print(f"Removing {duplicates_count} duplicate rows...")
    df = df.drop_duplicates()
    print(f"✓ Duplicates removed")
else:
    print("✓ No duplicates found")


Duplicate rows found: 0
✓ No duplicates found


## Verification and Saving Artifacts

In [250]:
# Verification
print("\nCleaning Summary:")
rows_after = len(df)
print(f"  - Initial rows: {rows_before}")
print(f"  - Final rows: {rows_after}")
print(f"  - Rows removed: {rows_before - rows_after}")
print(f"  - Missing values handled: {sum([item['count'] for item in missing_data_report])}")
# Verify no missing values
total_missing = df.isnull().sum().sum()
print(f"  - Total missing values remaining: {total_missing}")
print(f"  - Outliers capped: {sum([item['count'] for item in outlier_report])}")
# Verify no duplicates
total_duplicates = df.duplicated().sum()
print(f"  - Duplicate rows remaining: {total_duplicates}")
print("="*60)

# SAVE ARTIFACT 1: q2_cleaned_data.csv
df.to_csv('output/q2_cleaned_data.csv', index=False)
print("\n✓ Saved: output/q2_cleaned_data.csv")

# SAVE ARTIFACT 2: q2_cleaning_report.txt
with open('output/q2_cleaning_report.txt', 'w') as f:
    f.write("DATA CLEANING REPORT\n")
    f.write("=" * 60 + "\n\n")
    
    f.write(f"Rows before cleaning: {rows_before}\n\n")
    
    # Missing Data Handling
    f.write("Missing Data Handling:\n")
    if missing_data_report:
        for item in missing_data_report:
            f.write(f"- {item['column']}: {item['count']} missing values ({item['percentage']:.2f}%)\n")
            f.write(f"  Method: {item['method']}\n")
            f.write(f"  Result: All missing values filled\n\n")
    else:
        f.write("- No missing data detected\n\n")
    
    # Outlier Handling
    f.write("Outlier Handling:\n")
    if outlier_report:
        for item in outlier_report:
            f.write(f"- {item['column']}: Detected {item['count']} outliers using IQR method (3×IQR)\n")
            f.write(f"  Method: {item['method']}\n")
            f.write(f"  Bounds: [{item['lower_bound']:.2f}, {item['upper_bound']:.2f}]\n")
            f.write(f"  Result: {item['count']} values capped\n\n")
    else:
        f.write("- No outliers detected\n\n")
    
    # Duplicates
    f.write(f"Duplicates Removed: {duplicates_count}\n\n")
    
    # Data Type Conversions
    f.write("Data Type Conversions:\n")
    for conversion in data_type_conversions:
        f.write(f"- {conversion}\n")
    
    f.write(f"\nRows after cleaning: {rows_after}\n")

print("✓ Saved: output/q2_cleaning_report.txt")

# SAVE ARTIFACT 3: q2_rows_cleaned.txt
with open('output/q2_rows_cleaned.txt', 'w') as f:
    f.write(str(rows_after))

print("✓ Saved: output/q2_rows_cleaned.txt")


Cleaning Summary:
  - Initial rows: 195892
  - Final rows: 195892
  - Rows removed: 0
  - Missing values handled: 378901
  - Total missing values remaining: 0
  - Outliers capped: 102166
  - Duplicate rows remaining: 0

✓ Saved: output/q2_cleaned_data.csv
✓ Saved: output/q2_cleaning_report.txt
✓ Saved: output/q2_rows_cleaned.txt
